In [ ]:
import google.generativeai as genai

genai.configure(api_key="apikey")

model = genai.GenerativeModel("gemini-2.5-flash")
patient_data = {
    "age": 62,
    "gender": "female",
    "symptoms": ["vision loss", "headache"],
    "scan_findings": "MRI suggests meningioma",
    "provisional_diagnosis": "benign meningioma"
}
prompt = f"""
(condition_keyword,icd10_code,icd10_description
glioma,C71.9,"Malignant neoplasm of brain, unspecified"
meningioma,D32.9,"Benign neoplasm of meninges, unspecified"
pituitary,D35.2,Benign neoplasm of pituitary gland
notumor,Z00.0,General medical examination without abnormal findings)

You are a medical AI system.
Analyze this patient data and generate:

1. Clinical summary  
2. ICD-10 code  
3. Reasoning for the diagnosis  
4. Suggested next steps  

Patient data:
{patient_data}
"""

response = model.generate_content(prompt)
model_output = response.text
print(model_output)
import json

output = {
    "input_patient_data": patient_data,
    "model_generated_text": model_output
}

with open("milestone3_output.json", "w") as f:
    json.dump(output, f, indent=2)


**1. Clinical Summary**
A 62-year-old female presents with symptoms of vision loss and headache. MRI findings suggest a meningioma, leading to a provisional diagnosis of benign meningioma.

**2. ICD-10 Code**
D32.9

**3. Reasoning for the Diagnosis**
The provisional diagnosis explicitly states "benign meningioma." The patient's MRI scan findings also "suggests meningioma." According to the provided medical lookup table, the condition keyword "meningioma" is directly associated with the ICD-10 code D32.9, which describes "Benign neoplasm of meninges, unspecified." The symptoms of vision loss and headache are consistent with a space-occupying lesion like a meningioma, which can exert pressure on surrounding brain tissue or cranial nerves.

**4. Suggested Next Steps**
*   **Neurological and Ophthalmological Consultation:** Detailed neurological examination to assess cranial nerve function and a comprehensive ophthalmological evaluation, including visual field testing, given the reported v

In [ ]:

import os
import json
import time
import re
from glob import glob
from datetime import datetime
from pathlib import Path



# ---------- Configuration ----------
API_KEY = "apikey"

# Example model names that may be available:
# "models/gemini-1.5-flash-latest" or "models/gemini-1.5-pro-latest"
# Check your available models if you see NotFound errors.
MODEL_NAME = os.environ.get("GEMINI_MODEL", "models/gemini-2.5-flash")

DATA_INPUT_DIR = Path("Milestone_3/data_input")
OUTPUT_DIR = Path("Milestone_3/model_output")
LOG_PATH = OUTPUT_DIR / "pipeline_log.jsonl"

# Make sure output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional reference to uploaded docx (provided by user)
PROJECT_DOCX_PATH = "/mnt/data/AI-Powered-Enhanced_EHR_Imaging_&_Documentation_System (1).docx"

# Rate limiting: seconds to wait after each API call (tune as needed)
SLEEP_AFTER_CALL = 0.5

# ---------- Helper functions ----------
def list_input_files():
    """List JSON files in the data_input folder."""
    files = sorted(DATA_INPUT_DIR.glob("*.json"))
    return files

def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def save_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def log_event(event: dict):
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(event, default=str) + "\n")

def build_prompt(patient: dict):
    """Return a clear, constrained prompt for Gemini that requests JSON output."""
    # Keep the instructions deterministic, ask for JSON only.
    prompt = f"""
You are a medical AI assistant. Given the patient record below, generate a structured JSON response ONLY.
Do NOT include any extra explanatory text. The JSON must contain these keys:
 - patient_id (string)
 - clinical_note (string) : a concise clinical summary (2-6 sentences)
 - icd10_code (string) : the most appropriate single ICD-10 code (prefer benign codes for benign findings)
 - icd10_description (string) : short text describing the code
 - recommended_steps (array of strings) : up to 6 practical next steps (tests, referrals, urgent actions)
 - reasoning (string) : short justification (1-3 sentences)

Patient record:
{json.dumps(patient, ensure_ascii=False)}
"""
    return prompt.strip()

# ---------- Gemini API client wrapper (using google-generativeai) ----------
def init_gemini(api_key):
    """Initialize google generative ai library (import locally to avoid hard failure)."""
    try:
        import google.generativeai as genai
    except Exception as e:
        raise RuntimeError("google-generativeai package not installed. Run: pip install google-generativeai") from e

    genai.configure(api_key=api_key)
    return genai

def call_gemini(genai, model_name: str, prompt: str, max_output_tokens=800):
    """
    Call Gemini model and return the raw text output.
    Uses generate_content-like interface via the higher-level wrapper provided earlier.
    """
    # Some SDKs provide .GenerativeModel(...).generate_content; we'll detect available APIs.
    # Use the most common: genai.generate or genai.create or GenerativeModel
    try:
        # Preferred: generative model object
        model = genai.GenerativeModel(model_name)
        resp = model.generate_content(prompt)
        # response text commonly at resp.text
        text = getattr(resp, "text", None) or getattr(resp, "output", None) or str(resp)
        return text
    except Exception:
        # Fallback to genai.generate (older/newer variations)
        try:
            resp = genai.generate(model=model_name, prompt=prompt, max_output_tokens=max_output_tokens)
            if isinstance(resp, dict):
                # many wrappers put text at resp["candidates"][0]["content"]
                candidates = resp.get("candidates") or []
                if candidates:
                    return candidates[0].get("content", "")
            # if resp has .text
            return getattr(resp, "text", str(resp))
        except Exception as ex:
            raise RuntimeError("Failed to call Gemini model. See inner exception.") from ex

# ---------- Response parsing ----------
def attempt_json_parse(text: str):
    """Try to extract JSON from model text. Return dict or None."""
    if not text:
        return None
    # Try to find the first {...} block and parse
    match = re.search(r"(\{[\s\S]*\})", text)
    if match:
        candidate = match.group(1)
        # sanitize common issues (trailing commas)
        candidate = re.sub(r",\s*}", "}", candidate)
        candidate = re.sub(r",\s*\]", "]", candidate)
        try:
            return json.loads(candidate)
        except Exception:
            pass
    # Next, try to parse as pure JSON if it's clean
    try:
        return json.loads(text)
    except Exception:
        pass
    return None

def fallback_extract_fields(text: str):
    """If JSON parse fails, do a lightweight extraction using regex heuristics."""
    out = {
        "clinical_note": None,
        "icd10_code": None,
        "icd10_description": None,
        "recommended_steps": [],
        "reasoning": None
    }
    # ICD-like code pattern e.g., D32.9 or C70.0 or I50.1
    icd_match = re.search(r"\b([A-Z][0-9]{1,2}(?:\.[0-9A-Za-z]{1,4})?)\b", text)
    if icd_match:
        out["icd10_code"] = icd_match.group(1)
    # simple heuristics for sections
    sections = re.split(r"\n{2,}", text)
    # assign first paragraph to clinical_note if lengthy
    if sections:
        out["clinical_note"] = sections[0].strip()
    # recommended steps: look for lines that start with digits or hyphens
    steps = re.findall(r"(?:\n|^)[\-\*\d\)\. ]{0,3}\s*(Assess.*|Refer.*|Schedule.*|Order.*|Start.*|Consider.*|Obtain.*|Repeat.*)", text, flags=re.I)
    if steps:
        out["recommended_steps"] = [s.strip() for s in steps][:6]
    # reasoning: find a short sentence containing 'because' or 'since' or 'supported by'
    reasoning_match = re.search(r"([^.]{10,200}\b(?:because|since|supported by|due to)\b[^.]{0,200}\.)", text, flags=re.I)
    if reasoning_match:
        out["reasoning"] = reasoning_match.group(1).strip()
    return out

# ---------- Main pipeline ----------
def process_file(genai, file_path: Path):
    """Process a single patient JSON file and write output JSON."""
    patient = load_json(file_path)
    patient_id = patient.get("patient_id") or file_path.stem
    event = {
        "patient_file": str(file_path),
        "start": datetime.utcnow().isoformat()
    }
    try:
        prompt = build_prompt(patient)
        raw = call_gemini(genai, MODEL_NAME, prompt)
        time.sleep(SLEEP_AFTER_CALL)  # rate limit guard

        parsed = attempt_json_parse(raw)
        if parsed is None:
            parsed = fallback_extract_fields(raw)
            parsed["patient_id"] = patient_id
            parsed["model_raw_output"] = raw
            parsed["parse_method"] = "heuristic"
        else:
            # ensure patient_id present
            parsed.setdefault("patient_id", patient_id)
            parsed["parse_method"] = "json"

        # Add metadata
        parsed["_meta"] = {
            "source_file": str(file_path),
            "model_name": MODEL_NAME,
            "timestamp_utc": datetime.utcnow().isoformat()
        }

        # Save output
        out_path = OUTPUT_DIR / f"{patient_id}_output.json"
        save_json(parsed, out_path)

        event.update({
            "end": datetime.utcnow().isoformat(),
            "status": "success",
            "output_file": str(out_path),
            "raw_response_truncated": raw[:1000]  # avoid huge log lines
        })
        log_event(event)
        print(f"[OK] Processed {file_path.name} -> {out_path.name}")
        return out_path

    except Exception as e:
        event.update({
            "end": datetime.utcnow().isoformat(),
            "status": "error",
            "error": str(e)
        })
        log_event(event)
        print(f"[ERROR] {file_path.name}: {e}")
        return None

def main():
    genai = init_gemini(API_KEY)
    files = list_input_files()
    if not files:
        print("No input files found in", DATA_INPUT_DIR.resolve())
        return

    print(f"Found {len(files)} input files. Using model: {MODEL_NAME}")
    for f in files:
        process_file(genai, f)

if __name__ == "__main__":
    main()


Found 3 input files. Using model: models/gemini-2.5-flash


C:\Users\vigne\AppData\Local\Temp\ipykernel_7412\2352197341.py:166: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "start": datetime.utcnow().isoformat()
C:\Users\vigne\AppData\Local\Temp\ipykernel_7412\2352197341.py:188: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc": datetime.utcnow().isoformat()
C:\Users\vigne\AppData\Local\Temp\ipykernel_7412\2352197341.py:196: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "end": datetime.utcnow().isoformat(),


[OK] Processed patient1.json -> P001_output.json
[OK] Processed patient2.json -> P002_output.json
[OK] Processed sample_patient.json -> P003_output.json


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ref = ["MRI shows benign meningioma. Patient reports headache and vision loss."]
gen = ["A female patient with headache and vision loss. MRI suggests benign meningioma."]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(ref + gen)

score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
print("Cosine Similarity:", round(score, 3))



Cosine Similarity: 0.62
